# Vision pipeline accuracy: run with real Gemini and Hugging Face keys

I used this Colab notebook for the second run of `verify_vision_accuracy.py`, the one with real Gemini and Hugging Face keys (results are in `docs/PROJECT_REPORT.md` §4.1). It clones the repo, installs Tesseract, swaps in a debug-logging version of `verify_vision_accuracy.py`, and runs the four-stage pipeline (OCR, Gemini, Hugging Face, color fallback) on the 260 test images. The `labels.csv` fix and the OCR newline fix were already on GitHub when I ran it.

Run each cell from top to bottom. Keys are typed into a hidden prompt, so they are not printed or saved in the notebook file.


### 1. Clone the repo


In [ ]:
!git clone https://github.com/dhwanisinghal-sudo/Grocery-Recommender-System-Collaborative-Filtering-CV.git repo
%cd repo


### 2. Install dependencies (Tesseract OCR + Python packages)

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install streamlit pandas numpy scikit-learn scipy pillow requests pytesseract


### 3. Upload the one patched file
**Only `verify_vision_accuracy_patched.py` is needed this time** — `app.py` and the test-set images are already fixed on GitHub, so cell 1 already cloned the correct versions.

In [ ]:
from google.colab import files
import shutil, os

print('Choose verify_vision_accuracy_patched.py:')
up = files.upload()
fname = list(up.keys())[0]

os.makedirs('experiments', exist_ok=True)
shutil.move(fname, 'experiments/verify_vision_accuracy.py')
print('Replaced experiments/verify_vision_accuracy.py')


### 4. Enter your API keys (hidden input — not saved in this notebook file)

In [ ]:
import getpass, os

gemini_key = getpass.getpass('Paste GEMINI_API_KEY: ')
hf_key     = getpass.getpass('Paste HF_API_TOKEN: ')

os.makedirs('.streamlit', exist_ok=True)
with open('.streamlit/secrets.toml', 'w') as f:
    f.write(f'GEMINI_API_KEY = "{gemini_key}"\n')
    f.write(f'HF_API_TOKEN = "{hf_key}"\n')

del gemini_key, hf_key
print('secrets.toml written.')


### 5. Run the vision accuracy script
Calls the real Gemini + HF APIs for images OCR can't resolve. This version paces itself (2.5s between images) to avoid burning through free-tier rate limits, and prints *why* Gemini failed on any image it can't resolve instead of silently falling through — watch the console for lines like `Gemini did not resolve -- [...]`.

Takes roughly 20 minutes for 260 images (slower than before, on purpose).

In [ ]:
!python experiments/verify_vision_accuracy.py


### 6. Download the results

In [ ]:
from google.colab import files
files.download('experiments/vision_accuracy_results.csv')


### 7. (Recommended) Clear your keys from this session

In [ ]:
import os
if os.path.exists('.streamlit/secrets.toml'):
    os.remove('.streamlit/secrets.toml')
print('secrets.toml removed from this Colab session.')
